# MNIST Digits Recognition - MLOps 

## Kubeflow pipeline - Manual (without Kale)

In [2]:
import os
import kfp
from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics, ClassificationMetrics
import numpy as np
from tensorflow import keras
import tensorflow as tf
import glob
from sklearn.metrics import confusion_matrix

from kubernetes import client 
from kserve import KServeClient
from kserve import constants
from kserve import utils
from kserve import V1beta1InferenceService
from kserve import V1beta1InferenceServiceSpec
from kserve import V1beta1PredictorSpec
from kserve import V1beta1TFServingSpec

from datetime import datetime
import time
import subprocess
import sys

2024-11-12 10:52:48.483834: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-12 10:52:48.485450: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-11-12 10:52:48.516183: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-11-12 10:52:48.517203: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-12 10:52:49.203311: W tensorflow/compiler/tf2t

### Environment variables configuration

In [3]:
# Specify proxy and no_proxy value if applicable
http_proxy="" ## "http://<proxy_host>:<port>"
# After default.svc append comma separated list of IP address, hostname, CIDR block, service name, etc.
no_proxy="" ## "localhost,.cluster.local,.svc,.default.svc, <host>:<port>, x.x.x.x/y,..."

### Directories

In [4]:
import kfp
from pathlib import Path
kfp_client = kfp.Client()
user_mounted_dir_name = kfp_client.get_user_namespace()
user_shared_dir = f"{str(Path.home())}/{user_mounted_dir_name}/"
user_shared_dir

/opt/conda/lib/python3.11/site-packages/kfp/client/client.py:159: FutureWarning: This client only works with Kubeflow Pipeline v2.0.0-beta.2 and later versions.
  warnings.warn(


'/home/christian.temporale-hpe.com/christian-temporale-hpe-com-4bdac2aa/'

In [12]:
training_data_subdir = "training/"
model_subdir = "model-m"
model_trained_subdir = "model_trained_m"
model_export_subdir = "model_exported-m"

### Hyperparameters

In [6]:
num_epochs = 1
lr = 0.1

## Preprocessing

In [7]:
final_training_data_dir = user_shared_dir + training_data_subdir

x_train_artifact = final_training_data_dir + "train-images-pickle.npy"
x_test_artifact = final_training_data_dir + "test-images-pickle.npy"
y_train_artifact = final_training_data_dir + "train-labels-pickle.npy"
y_test_artifact = final_training_data_dir + "test-labels-pickle.npy"

x_train_preproc = final_training_data_dir + "xtrain-preproc.npy"
x_test_preproc = final_training_data_dir + "xtest-preproc.npy"

# load data artifact store
x_train = np.load(x_train_artifact) 
x_test = np.load(x_test_artifact)
    
# reshaping the data
# reshaping pixels in a 28x28px image with greyscale, canal = 1. This is needed for the Keras API
x_train = x_train.reshape(-1,28,28,1)
x_test = x_test.reshape(-1,28,28,1)
# normalizing the data
# each pixel has a value between 0-255. Here we divide by 255, to get values from 0-1
x_train = x_train / 255
x_test = x_test / 255
    
#logging metrics using Kubeflow Artifacts
print("Len x_train", x_train.shape[0])
print("Len y_train", x_test.shape[0])
      
# save feuture in artifact store
np.save(x_train_preproc, x_train)
#os.rename("tmp/x_train.npy", x_train_processed.path)
    
np.save(x_test_preproc, x_test)

Len x_train 60000
Len y_train 10000


## Model build

In [10]:
final_model_data_dir = user_shared_dir + model_subdir

if not os.path.exists(final_model_data_dir):
    os.makedirs(final_model_data_dir, 0o777)
    
model = keras.models.Sequential()
model.add(keras.layers.Conv2D(64, (3, 3), activation='relu', input_shape=(28,28,1)))
model.add(keras.layers.MaxPool2D(2, 2))

model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(64, activation='relu'))
model.add(keras.layers.Dense(32, activation='relu'))

model.add(keras.layers.Dense(10, activation='softmax'))
    
#saving model
model.save(final_model_data_dir)

Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.
Assets written to: /home/christian.temporale-hpe.com/christian-temporale-hpe-com-4bdac2aa/model-m/assets


## Training

In [15]:
model_trained_dir = user_shared_dir + model_trained_subdir
model_export_dir = user_shared_dir + model_export_subdir

if not os.path.exists(model_trained_dir):
    os.makedirs(model_trained_dir, 0o777)

if not os.path.exists(model_export_dir):
    os.makedirs(model_export_dir, 0o777)
    
#load dataset
x_train = np.load(x_train_preproc)
x_test = np.load(x_test_preproc)
y_train = np.load(y_train_artifact)
y_test = np.load(y_test_artifact)
    
#load model structure
model = keras.models.load_model(final_model_data_dir)
      
#compile the model - we want to have a binary outcome
model.compile(tf.keras.optimizers.SGD(learning_rate=lr),
              loss="sparse_categorical_crossentropy",
              metrics=['accuracy'])

    
#fit the model and return the history while training
history = model.fit(
      x = x_train,
      y = y_train,
      epochs = num_epochs,
      batch_size = 20,
)

     
# Test the model against the test dataset
# Returns the loss value & metrics values for the model in test mode.
model_loss, model_accuracy = model.evaluate(x=x_test,y=y_test)

print(f"model_loss={model_loss} model_accuracy={model_accuracy} ")


#build a confusione matrix
y_predict = model.predict(x=x_test)
y_predict = np.argmax(y_predict, axis=1)
print(f"y_predict={y_predict}")
    
#adding /1/ subfolder for TFServing
model_trained_dir = model_trained_dir + '/1/'
print("model_trained_uri: " + model_trained_dir)

model_export_dir = model_export_dir + '/1/'
print("model_export_uri: " + model_export_dir)

# save model in local storage
keras.models.save_model(model, model_trained_dir, save_format='tf')

# save model in model export volume
keras.models.save_model(model, model_export_dir, save_format='tf')

No training configuration found in save file, so the model was *not* compiled. Compile it manually.


313/313 [==============================] - 2s 6ms/step - loss: 0.0737 - accuracy: 0.9752
model_loss=0.0737297385931015 model_accuracy=0.9751999974250793 
313/313 [==============================] - 2s 6ms/step
y_predict=[7 2 1 ... 4 5 6]
model_trained_uri: /home/christian.temporale-hpe.com/christian-temporale-hpe-com-4bdac2aa/model_trained_m/1/
model_export_uri: /home/christian.temporale-hpe.com/christian-temporale-hpe-com-4bdac2aa/model_exported-m/1/


Assets written to: /home/christian.temporale-hpe.com/christian-temporale-hpe-com-4bdac2aa/model_trained_m/1/assets
Assets written to: /home/christian.temporale-hpe.com/christian-temporale-hpe-com-4bdac2aa/model_exported-m/1/assets


## Validation

In [16]:
#Load model from the volume
model = keras.models.load_model(model_trained_dir)

test_accu = model.evaluate(x_test, y_test)
print('The testing accuracy is :',test_accu[1]*100, '%')

313/313 [==============================] - 2s 6ms/step - loss: 0.0737 - accuracy: 0.9752
The testing accuracy is : 97.51999974250793 %


## Model serving

In [ ]:
#from kale.sdk import step
import subprocess

model_dir = "/mnt/export/"
model_volume = "user-pvc"  
model_name = "digitrec-serve"

api_version = 'serving.kserve.io/v1beta1'

serving_command = f"""
     echo '
      apiVersion: "{api_version}"
      kind: "InferenceService"
      metadata:
        name: {model_name}
        annotations:
          "sidecar.istio.io/inject": "false"
      spec:
        predictor:
          tensorflow:
            storageUri: "pvc://{model_volume}/model_trained_m/"
            resources:
              limits:
                cpu: '2'
                memory: 1Gi
              requests:
                cpu: 100m
                memory: 50Mi'  | kubectl create -f -
      """

# Run the serving command
result = subprocess.run(serving_command, shell=True, check=True)
    
# Check if the serving container is running successfully
if result.returncode == 0:
    print("TensorFlow Serving started successfully.")
else:
    raise Exception("Failed to start TensorFlow Serving.")

inferenceservice.serving.kserve.io/digitrec-serve created
TensorFlow Serving started successfully.
